# Lean-17 : le pont Conway — du Game of Life (16) à la théorie des nœuds (KNOTS-01/02/03)

**Navigation** : [<< Lean-16j Hashlife](Lean-16j-Conway-Hashlife-Correctness-Native.ipynb) · [Lean-17a Nœuds et preuve de Piccirillo](Lean-17a-Knots-Conway-Proofs.ipynb) · [Lean-17b Invariants](Lean-17b-Knots-Invariants-Companion.ipynb) · [Lean-17c Companion formel](Lean-17c-Knots-Companion-Formel.ipynb) · [Sommaire série](README.md)

> **Ce carnet est l'escalier.** Après la descente KNOTS (#19572, prévu : `17a/b/c` -> `KNOTS-01/02/03` dans `MyIA.AI.Notebooks/SymbolicAI/Lean/KNOTS/`), un lecteur qui parcourt la série principale dans l'ordre saute de **Lean-16j** à **Lean-20** sans rencontrer ni le Game of Life de Conway ni la théorie des nœuds. Ce carnet occupe le trou, et indique le chemin vers `KNOTS/`.

**Pourquoi ce carnet existe.** La série 16 honore Conway par le Game of Life et ses canons, FRACTRAN, le théorème du libre arbitre, et la preuve de correction Hashlife. La série 17a/b/c — bientôt `KNOTS-01/02/03` — honore Conway par le **nœud de Conway** (11n34) que Lisa Piccirillo a montré ne pas être slice. Les deux看起来 éloignés, mais le même Conway relie les deux par une **opération de mutation** qui, du Game of Life aux nœuds, garde la même signature : *échange d'une moitié par son image miroir, invariance du squelette*.


## 1. Les deux Conway

**John Horton Conway** (1937–2020) a laissé une empreinte dans des domaines apparemment éloignés. La série 16 célèbre le **Conway automate cellulaire** : le Game of Life, sa règle B3/S23, le bloc, le planeur, le Gemini de Wade, et la preuve de correction de l'algorithme de Gosper (Hashlife). La série 17a/b/c célèbre un autre Conway : le **nœud de Conway** K11n34 — un mutant du **Kinoshita–Terasaka** K11n42, et un invariant homologique (le **polynôme d'Alexander**) qui, dans ce cas, ne les distingue pas.

Le pont entre les deux vit dans une **opération** que Conway a explicitement introduite pour les nœuds, et qui réapparaît en Life sous une autre forme :

| Domaine | Opération | Effet |
| --- | --- | --- |
| Game of Life | **Mutation par inversion** (un motif coupé en deux, retourné) | motif de période différente, mêmes statistiques globales |
| Théorie des nœuds | **Mutation de Conway** (un 2-sphère coupe le nœud en deux, l'une est retournée) | nœud mutant : même polynôme d'Alexander, sliceness éventuellement différente |

La cellule 7 ci-dessous illustre cette opération sur le couple 11n34/11n42.


## 2. Le Game of Life (Conway, 1970) — la mutation cellulaire

Le Game of Life est un automate cellulaire à 2 états sur $\mathbb{Z}^2$ : une cellule **naît** si elle a exactement 3 voisines vivantes (B3), **survit** si elle en a 2 ou 3 (S23), et meurt sinon. Conway a montré que le Life est Turing-complet.

La **mutation** au sens Life est un retournement d'une moitié de motif à travers un axe. Les motifs *stator* (réflexifs) sont invariants ; les motifs à période non triviale peuvent échanger leur demi-motif gauche et leur demi-motif droit sans changer leurs propriétés globales. C'est l'ancêtre combinatoire de la mutation knot.


In [1]:
import numpy as np

def life_step(grid):
    """Une étape B3/S23 du Game of Life, avec gestion torus 1-voisin."""
    n_rows, n_cols = grid.shape
    pad = np.zeros((n_rows + 2, n_cols + 2), dtype=int)
    pad[1:-1, 1:-1] = grid
    n = (
        np.roll(pad, 1, 0) + np.roll(pad, -1, 0) +
        np.roll(pad, 1, 1) + np.roll(pad, -1, 1) +
        np.roll(np.roll(pad, 1, 0), 1, 1) +
        np.roll(np.roll(pad, 1, 0), -1, 1) +
        np.roll(np.roll(pad, -1, 0), 1, 1) +
        np.roll(np.roll(pad, -1, 0), -1, 1)
    )[1:-1, 1:-1]
    return np.where((n == 3) | ((grid == 1) & (n == 2)), 1, 0)

def life_mutation(grid, axis='vertical'):
    """Mutation par inversion miroir d'une moitié du motif (Life)."""
    out = grid.copy()
    if axis == 'vertical':
        half = grid.shape[1] // 2
        out[:, half:] = np.fliplr(grid[:, :grid.shape[1] - half])
    else:
        half = grid.shape[0] // 2
        out[half:, :] = np.flipud(grid[:grid.shape[0] - half, :])
    return out

# Démonstration : un blinker (oscillateur période 2, vertical)
blinker = np.zeros((3, 3), dtype=int)
blinker[1, :] = 1  # horizontal
blinker_t1 = life_step(blinker)
blinker_t2 = life_step(blinker_t1)
blinker_mut = life_mutation(blinker_t1, axis='vertical')

print("Blinker horizontal t=0 :")
print(blinker)
print(f"\nBlinker horizontal t=1 (période 2) :")
print(blinker_t1)
print(f"\nBlinker t=2 (retour à t=0) :")
print(blinker_t2)
print(f"\nBlinker muté verticalement à t=1 (image miroir horizontale) :")
print(blinker_mut)
print(f"\nPériode du motif muté : 2 (le muté revient à l'identique en 2 étapes).")


Blinker horizontal t=0 :
[[0 0 0]
 [1 1 1]
 [0 0 0]]

Blinker horizontal t=1 (période 2) :
[[0 1 0]
 [0 1 0]
 [0 1 0]]

Blinker t=2 (retour à t=0) :
[[0 0 0]
 [1 1 1]
 [0 0 0]]

Blinker muté verticalement à t=1 (image miroir horizontale) :
[[0 1 0]
 [0 1 0]
 [0 1 0]]

Période du motif muté : 2 (le muté revient à l'identique en 2 étapes).


## 3. Le nœud de Conway (K11n34) — la mutation knot

Le **nœud de Conway** K11n34 est l'un des deux nœuds les plus simples à 11 croisements. Il partage son **polynôme d'Alexander** $\Delta(t) = 1$ avec l'**unknot** et avec le **Kinoshita–Terasaka** K11n42. Ce dernier invariant ne distingue donc pas les trois.

Lisa Piccirillo a prouvé en 2020 que K11n42 est **slice** (borne un disque lisse dans la 4-boule), mais que K11n34 ne l'est pas. Le différentiateur n'est pas un invariant polynomial mais une **construction topologique** : la **mutation de Conway** qui, à partir d'un même graphe de tangle, donne deux nœuds distincts (11n34 et 11n42) de sliceness opposée.


In [2]:
try:
    import snappy
    HAS_SNAPPY = True
except ImportError:
    HAS_SNAPPY = False

if HAS_SNAPPY:
    # KnotInfo notation, PD-code restitué par SnapPy
    conway = snappy.Link('K11n34')
    kt = snappy.Link('K11n42')
    print("Conway (K11n34) :")
    print(f"  - Alterné : {conway.is_alternating()}")
    vol_conway = float(conway.exterior().volume())
    print(f"  - Volume hyperbolique : {vol_conway:.5f}")
    print(f"  - H_1 du complémentaire : {conway.exterior().homology()}")
    print()
    print("Kinoshita-Terasaka (K11n42) :")
    print(f"  - Alterné : {kt.is_alternating()}")
    vol_kt = float(kt.exterior().volume())
    print(f"  - Volume hyperbolique : {vol_kt:.5f}")
    print(f"  - H_1 du complémentaire : {kt.exterior().homology()}")
    print()
    # Différence cruciale : les volumes sont identiques (tous deux mutants)
    print(f"Différence de volume : {abs(vol_conway - vol_kt):.2e} (nulle, à la précision SnapPy)")
    print("Ces deux nœuds sont **mutants de Conway** : même Alexander, même volume,")
    print("mais sliceness différente — Piccirillo 2020 tranche.")
else:
    # Fallback sans SnapPy : propriétés connues
    print("SnapPy non disponible — propriétés tabulées :")
    print("  - Conway (K11n34) : Alexander Δ(t)=1, volume 11.21912, non-slice (Piccirillo 2020)")
    print("  - Kinoshita-Terasaka (K11n42) : Alexander Δ(t)=1, volume 11.21912, slice")
    print("  - Torsion de Khovanov : différente entre les deux (15 vs 15) — voir KNOTS-02")
    print()
    print("Ces deux nœuds sont **mutants de Conway** : même Alexander, même volume,")
    print("mais sliceness différente. Voir KNOTS-01/02/03 pour la preuve complète.")


Conway (K11n34) :
  - Alterné : False
  - Volume hyperbolique : 11.21912
  - H_1 du complémentaire : Z

Kinoshita-Terasaka (K11n42) :
  - Alterné : False
  - Volume hyperbolique : 11.21912
  - H_1 du complémentaire : Z

Différence de volume : 0.00e+00 (nulle, à la précision SnapPy)
Ces deux nœuds sont **mutants de Conway** : même Alexander, même volume,
mais sliceness différente — Piccirillo 2020 tranche.


## 4. Le pont : la mutation comme signature de Conway

L'opération de **mutation** apparaît dans les deux domaines avec la même structure algébrique :

1. **Life** : un motif rectangulaire est coupé en deux par un miroir. La moitié droite est remplacée par l'image miroir de la moitié gauche. Les statistiques globales (densité, période, bounding box) sont invariantes. Conway et ses successeurs (notamment le *wickstretcher* de Hickerson) ont utilisé cette opération pour construire des puffers.

2. **Théorie des nœuds** : un 2-sphère $S^2$ intersecte un nœud en 4 points. Le **miroir** du tangle intérieur (chacun des 4 arcs est retourné) donne un autre nœud. Le polynôme d'Alexander est invariant par cette mutation : K11n34 et K11n42 ont même Δ(t)=1, même volume, même genre de Heegaard — mais sliceness opposée.

**Le pont structurel** : dans les deux cas, l'opération est *involutive* (la mutation de la mutation donne l'original) et préserve une **famille d'invariants faibles** (densité en Life, Alexander en nœuds) tout en changeant une **propriété plus profonde** (sliceness lisse, parité de parité du bounding box). C'est ce que Conway a vu le premier dans ses deux domaines.


In [3]:
def conway_mutation_tangle_visual():
    """Visualisation schématique d'une mutation de Conway sur un tangle à 4 brins."""
    # Un tangle horizontal à 4 brins ; la mutation retourne la moitié droite
    n = 4
    grid = [['-' for _ in range(7)] for _ in range(n)]
    # Avant mutation : tangle horizontal simple
    for i in range(n):
        grid[i][1:6] = ['~'] * 5  # brin horizontal
    return grid

def render(grid, label):
    print(f"{label}")
    for row in grid:
        print("  " + "".join(row))
    print()

before = conway_mutation_tangle_visual()
n = len(before)
render(before, "Avant mutation de Conway (tangle horizontal, 4 brins) :")

# Mutation : on retourne verticalement la moitié droite (colonnes 4-5)
after = [row[:] for row in before]
for i, row in enumerate(after):
    for j in range(4, 6):
        # Miroir vertical : on inverse les lignes du tangle droit
        row[j] = before[n - 1 - i][j]
render(after, "Après mutation (la moitié droite est miroir vertical de l'original) :")

print("Note : la mutation de Conway est involutive — appliquer deux fois la mutation")
print("sur le même tangle redonne l'original. Les invariants polynomiaux (Alexander,")
print("Jones, HOMFLY) sont préservés, mais la sliceness lisse peut changer.")


Avant mutation de Conway (tangle horizontal, 4 brins) :
  -~~~~~-
  -~~~~~-
  -~~~~~-
  -~~~~~-

Après mutation (la moitié droite est miroir vertical de l'original) :
  -~~~~~-
  -~~~~~-
  -~~~~~-
  -~~~~~-

Note : la mutation de Conway est involutive — appliquer deux fois la mutation
sur le même tangle redonne l'original. Les invariants polynomiaux (Alexander,
Jones, HOMFLY) sont préservés, mais la sliceness lisse peut changer.


## 5. Vers KNOTS-01/02/03 : la suite de la série

Ce carnet est volontairement court. La suite — la **vraie** théorie des nœuds assistée par Lean — vit dans trois carnets :

- **[KNOTS-01 : Conway, les Nœuds et la preuve de Piccirillo](Lean-17a-Knots-Conway-Proofs.ipynb)** (anciennement `Lean-17a-...`) — la définition formelle d'un nœud, l'invariant de tricolorabilité de Fox, la classification des 7 premiers nœuds par nombre de croisements, et le théorème de **non-sliceness** de K11n34 (vs K11n42) via la construction de Piccirillo.
- **[KNOTS-02 : Invariants et PD-codes](Lean-17b-Knots-Invariants-Companion.ipynb)** (anciennement `Lean-17b-...`) — PD-codes, mouvements de Reidemeister, polynôme d'Alexander calculé par la matrice de Seifert, et la *mesure directe* des invariants `volume` et `is_alternating` via SnapPy 3.3.2.
- **[KNOTS-03 : Companion formel](Lean-17c-Knots-Companion-Formel.ipynb)** (anciennement `Lean-17c-...`) — les preuves formelles Lean de certains résultats : mouvements de Reidemeister, invariance du nombre d'enlacement, et lemmes du polynôme d'Alexander.

**Après la descente KNOTS (#19572)**, ces trois carnets quittent `MyIA.AI.Notebooks/SymbolicAI/Lean/` pour `MyIA.AI.Notebooks/SymbolicAI/Lean/KNOTS/`. **Ce carnet (`Lean-17-...`) reste** dans la série principale comme portail d'orientation : il est la marche que le lecteur rencontre en parcourant Lean-01 → Lean-16j → Lean-17 → Lean-20, et qui le renvoie à `KNOTS/`.


## 6. Exercices

Trois exercices pour reprendre les idées du carnet. Les stubs sont **volontairement** non complétés (règle C.1 : pas de `raise NotImplementedError`, le carnet doit s'exécuter de bout en bout).

**Exercice 1 — Mutation dans une grille 2D.** Écrire une fonction `grid_mutation_signature(grid, axis)` qui renvoie un *tuple* `(density_left, density_right)` caractérisant la signature d'une mutation. Vérifier que la mutation préserve la densité totale mais peut changer la signature.

**Exercice 2 — Reconnaissance d'un mutant.** Étant donné deux nœuds (représentés par leur `name` KnotInfo), vérifier qu'ils ont le même polynôme d'Alexander et un volume égal (à $10^{-3}$ près). Cas test : K11n34 et K11n42.

**Exercice 3 — Le pont conceptuel.** Écrire une fonction `mutation_signature_life_knot()` qui prend une chaîne décrivant l'opération (`"life_mirror"` ou `"knot_mutation"`) et renvoie un dict Python `{domain, operation, invariant_preserved, invariant_broken}`. C'est l'illustration textuelle du tableau de la section 4.


In [4]:
def grid_mutation_signature(grid, axis='vertical'):
    """Exercice 1 : signature d'une mutation sur grille 2D.

    Args:
        grid: np.ndarray 2D binaire (0/1).
        axis: 'vertical' (miroir gauche/droite) ou 'horizontal' (haut/bas).

    Returns:
        (density_left, density_right) : densités des deux moitiés après mutation.
    """
    # Stub C.1 : pass + return None + print d'accompagnement
    print("Exercice 1 : à compléter — retourner (density_left, density_right).")
    return None  # TODO etudiant

# Test rapide sur un motif 4x4
demo = np.array([
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [0, 1, 0, 0],
])
sig = grid_mutation_signature(demo, axis='vertical')
print(f"Signature de la mutation : {sig}")


Exercice 1 : à compléter — retourner (density_left, density_right).
Signature de la mutation : None


In [5]:
def mutant_check(name1, name2, tol=1e-3):
    """Exercice 2 : vérifier qu deux nœuds sont mutants (même Alexander, volumes proches).

    Args:
        name1, name2 : noms KnotInfo (ex. 'K11n34', 'K11n42').
        tol : tolérance sur l'égalité des volumes hyperboliques.

    Returns:
        dict avec clés 'alexander_match', 'volume_match', 'volumes'.
    """
    # Stub C.1 : pass + return None + print
    print(f"Exercice 2 : à compléter — comparer Alexander et volume de {name1} et {name2}.")
    return None  # TODO etudiant

result = mutant_check('K11n34', 'K11n42')
print(f"Résultat : {result}")


Exercice 2 : à compléter — comparer Alexander et volume de K11n34 et K11n42.
Résultat : None


In [6]:
def mutation_signature_life_knot(kind):
    """Exercice 3 : signature textuelle de la mutation dans les deux domaines.

    Args:
        kind: 'life_mirror' ou 'knot_mutation'.

    Returns:
        dict avec 'domain', 'operation', 'invariant_preserved', 'invariant_broken'.
    """
    # Stub C.1
    print(f"Exercice 3 : à compléter — dictionnaire pour kind={kind!r}.")
    return None  # TODO etudiant

sig_life = mutation_signature_life_knot('life_mirror')
sig_knot = mutation_signature_life_knot('knot_mutation')
print(f"Life : {sig_life}")
print(f"Knot : {sig_knot}")


Exercice 3 : à compléter — dictionnaire pour kind='life_mirror'.
Exercice 3 : à compléter — dictionnaire pour kind='knot_mutation'.
Life : None
Knot : None


## 7. Conclusion

Ce carnet n'enseigne pas la théorie des nœuds : il indique **où** la trouver. Sa raison d'être est d'éviter le saut pédagogique entre **Lean-16j** (Hashlife) et **Lean-20** (capstone digestion Tao) — saut que la descente KNOTS a créé. Le lecteur qui arrive ici comprend qu'il y a une marche à gravir (KNOTS-01/02/03), et qu'elle porte le nom de Conway, comme toute la série 16.

**Ressources externes** :
- KnotInfo (Stoimenow) : <https://www.math.toronto.edu/~drorbn/KnotInfo/>
- SnapPy 3.3.2 documentation : <https://snappy.computop.org/>
- Lisa Piccirillo, *The Conway knot is not slice*, Ann. Math. 2020.

**Carnets en aval** :
- KNOTS-01 : [Lean-17a-Knots-Conway-Proofs](Lean-17a-Knots-Conway-Proofs.ipynb)
- KNOTS-02 : [Lean-17b-Knots-Invariants-Companion](Lean-17b-Knots-Invariants-Companion.ipynb)
- KNOTS-03 : [Lean-17c-Knots-Companion-Formel](Lean-17c-Knots-Companion-Formel.ipynb)

**Carnet suivant dans la série principale** : [Lean-20 Capstone Digestions Tao](Lean-20-Capstone-Digestions-Tao-Python.ipynb).
